# 08b — Refresh Dashboard

Verifies the **Book Crossing Analysis** Lakeview dashboard is active and
its underlying `gold.*` tables are populated.

The dashboard queries live data on every view — no explicit data refresh
is needed. This notebook serves as a pipeline gate and convenience link.

In [0]:
import json, pathlib

SETUP_CONFIG = "/Workspace/Users/tharcisio.leone@x1f-fink.one/poc_recomender_model/config/00_setup.json"

with open(SETUP_CONFIG) as f:
    setup = json.load(f)

DASHBOARD_ID = setup.get("dashboard_id", "01f1c6f59dc218d2ba43f99cc6a82585")
WORKSPACE_URL = setup.get("workspace_url", "https://adb-7405609268959791.11.azuredatabricks.net")

print(f"Dashboard ID : {DASHBOARD_ID}")
print(f"Workspace URL: {WORKSPACE_URL}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import LifecycleState

w = WorkspaceClient()

dashboard = w.lakeview.get(DASHBOARD_ID)

if dashboard.lifecycle_state == LifecycleState.TRASHED:
    print(f"⚠ Dashboard '{dashboard.display_name}' ({DASHBOARD_ID}) is in Trash.")
    print("  Restore it from Workspace sidebar > Trash to re-enable.")
else:
    print(f"Dashboard '{dashboard.display_name}' is {dashboard.lifecycle_state.value}.")
    print(f"  Path         : {dashboard.path}")
    print(f"  Updated at   : {dashboard.update_time}")

row_count = spark.sql("SELECT COUNT(*) AS n FROM gold.book_ratings").first()["n"]
print(f"  gold.book_ratings rows: {row_count:,}")
print("\nDashboard datasets query live data — no explicit refresh needed.")

In [0]:
url = f"{WORKSPACE_URL}/sql/dashboards/{DASHBOARD_ID}"
print(f"Published dashboard URL:\n  {url}")

displayHTML(f'<a href="{url}" target="_blank" style="font-size:16px;">Open Book Crossing Analysis Dashboard</a>')